# 02 - Seasonal outlook: will the next 3 months be below normal?
At the end of each month we predict whether the next 3 months' rainfall falls in the **driest third** of history for that season.

**No-leakage rules** (enforced in `sentinel/outlook.py`, tested in `tests/test_outlook_leakage.py`):
1. Time split - the test set is every issue whose target starts in the last 8 complete years; training targets must end before that (purged gap).
2. All normals, standard deviations and tercile thresholds come from training months only.
3. ONI is the season ending the previous month (centred two months back) and DMI is last month's value - what was published at issue time.
4. Model type and warning threshold are chosen on the last 5 training years, never on the test years.
5. Outlooks are not issued for dry-season targets (3-month normal < 30 mm) where 'driest third' is meaningless.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sentinel import config as C
OUT, RAW = C.OUT_DIR, C.RAW_DIR
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
csv = lambda n, **k: pd.read_csv(OUT / n, **k)
summary = json.loads((OUT / "run_summary.json").read_text())
print("Daily data:", summary["daily_range"], "| built", summary["built_utc"])

In [ ]:
info = json.loads((OUT / "outlook_info.json").read_text()); print(json.dumps(info, indent=1))

## Per-county results on the test years
Baselines: **climatology** (always 33%; as a categorical forecast, a random 1-in-3 warning, so precision = observed base rate and recall = 1/3) and **persistence** (dry last 3 months -> predict dry).

In [ ]:
met = csv("outlook_metrics.csv")
cols = ["county", "n_test", "observed_below_rate", "model_precision", "model_recall", "model_f1", "persistence_precision",
        "persistence_recall", "persistence_f1", "climatology_f1", "brier_model", "brier_persistence", "brier_climatology",
        "bss_model_vs_clim", "bss_90ci_lo", "bss_90ci_hi", "bss_persistence_vs_clim", "roc_auc_model", "verdict"]
met[cols].round(3)

In [ ]:
m = met[met.county != "ALL (pooled)"].sort_values("bss_model_vs_clim")
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(m.bss_model_vs_clim, m.county, xerr=[m.bss_model_vs_clim - m.bss_90ci_lo, m.bss_90ci_hi - m.bss_model_vs_clim], fmt="o", capsize=3)
ax.scatter(m.bss_persistence_vs_clim, m.county, marker="x", color="#c0392b", label="persistence")
ax.axvline(0, color="k", lw=0.8); ax.set_xlabel("Brier skill score vs climatology (90% bootstrap CI)"); ax.legend()

## By target season (all counties pooled)
The Indian Ocean Dipole is a known driver of the Oct-Dec short rains, so skill is expected there before Mar-May.

In [ ]:
csv("outlook_by_season.csv").round(3)

## Reliability: are the probabilities honest?

In [ ]:
te = pd.read_parquet(OUT / "outlook_test_predictions.parquet")
bins = np.linspace(0, 1, 6); te["bin"] = pd.cut(te.p_model, bins, include_lowest=True)
rel = te.groupby("bin", observed=True).agg(forecast=("p_model", "mean"), observed=("y", "mean"), n=("y", "size"))
display(rel.round(3))
fig, ax = plt.subplots(figsize=(4.5, 4.5)); ax.plot([0, 1], [0, 1], "k:"); ax.plot(rel.forecast, rel.observed, "o-")
ax.axhline(1/3, color="grey", lw=0.6); ax.set_xlabel("forecast probability"); ax.set_ylabel("observed frequency")

## Plain statement of skill

In [ ]:
for r in met.itertuples():
    ci = f"BSS {r.bss_model_vs_clim:+.2f} (90% CI {r.bss_90ci_lo:+.2f} to {r.bss_90ci_hi:+.2f})"
    print(f"- {r.county}: {r.verdict}. {ci}; F1 model {r.model_f1:.2f} vs persistence {r.persistence_f1:.2f} vs climatology {r.climatology_f1:.2f}.")
ns = met[(met.county != "ALL (pooled)") & met.verdict.str.startswith("No skill")].county.tolist()
print(f"\nWhere there is no skill ({', '.join(ns) or 'none'}), the dashboard and agent publish the climatological 33% instead of the model.")

## Live outlook (issued at the end of the latest complete month)

In [ ]:
csv("outlook_live.csv")